In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Load dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
df = pd.read_excel(url)

# Data Cleaning
df = df.dropna(subset=['CustomerID'])  # Drop rows with missing CustomerID
df = df[df['Quantity'] > 0]  # Remove negative quantities (returns)

# Feature Engineering
df['TotalSpend'] = df['Quantity'] * df['UnitPrice']
customer_data = df.groupby('CustomerID').agg({
    'TotalSpend': 'sum',
    'InvoiceNo': 'nunique',
    'InvoiceDate': lambda x: (pd.to_datetime('2011-12-31') - x.max()).days
}).rename(columns={'InvoiceNo': 'PurchaseFrequency', 'InvoiceDate': 'Recency'})
customer_data['Country'] = df.groupby('CustomerID')['Country'].agg(lambda x: x.mode()[0])
customer_data = pd.get_dummies(customer_data, columns=['Country'], drop_first=True)
customer_data['TotalSpend'] = customer_data['TotalSpend'].clip(upper=customer_data['TotalSpend'].quantile(0.95))

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(customer_data)

# Data Exploration
plt.figure(figsize=(10, 6))
sns.scatterplot(x=customer_data['TotalSpend'], y=customer_data['PurchaseFrequency'])
plt.title('Customer Distribution by Total Spend and Purchase Frequency')
plt.xlabel('Total Spend (£)')
plt.ylabel('Purchase Frequency')
plt.savefig('customer_plot.png', bbox_inches='tight')
plt.close()

# Train and evaluate clustering models
models = {
    'K-Means': KMeans(n_clusters=4, random_state=42),
    'Agglomerative Hierarchical': AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'Gaussian Mixture Model': GaussianMixture(n_components=4, random_state=42)
}
results = []
for name, model in models.items():
    labels = model.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    db_score = davies_bouldin_score(X, labels)
    results.append({
        'Model': name,
        'Silhouette Score': sil_score,
        'Davies-Bouldin Index': db_score
    })
    if name == 'K-Means':
        plt.figure(figsize=(10, 6))
        sns.scatterplot(x=customer_data['TotalSpend'], y=customer_data['PurchaseFrequency'], hue=labels, palette='deep')
        plt.title('Customer Distribution by Total Spend and Purchase Frequency (K-Means Clusters)')
        plt.xlabel('Total Spend (£)')
        plt.ylabel('Purchase Frequency')
        plt.savefig('customer_plot.png', bbox_inches='tight')
        plt.close()

# Display results
results_df = pd.DataFrame(results)
print(results_df)

# Feature Importance (approximated via GMM component means)
gmm = models['Gaussian Mixture Model']
feature_importance = pd.Series(np.abs(gmm.means_).mean(axis=0), index=customer_data.columns).sort_values(ascending=False)[:10]
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance.values, y=feature_importance.index)
plt.title('Feature Contributions to Clustering (GMM)')
plt.xlabel('Mean Absolute Component Value')
plt.ylabel('Feature')
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.close()
